link to the learning path: https://chatgpt.com/c/691b22d4-96fc-8332-a155-66384fdf7ed4

https://chatgpt.com/c/691b5e8f-77fc-8326-8044-96e1139e33d2

# 1. Implement feed forward neural network with two layers (input and one hidden layer)

### We have a 2-layer neural network: Input → Hidden Layer → Output

what we need?

✔ A weight matrix from Input → Hidden\
✔ A weight matrix from Hidden → Output\
✔ Bias vectors for each layer

1. Step 1: W1 — weights from input to hidden layer (Inputs: n_inputs, Hidden: n_hidden)\
W1 shape = (n_inputs, n_hidden)\
self.W1 = 0.01 * np.random.randn(self.n_inputs, self.n_hidden)

2. Step 2: b1 — bias for hidden layer\
b1 shape = (1, n_hidden)\
self.b1 = np.zeros((1, self.n_hidden))

3. Step 3: W2 — weights from hidden → output\
W2 shape = (n_hidden, 1)\
self.W2 = 0.01 * np.random.randn(self.n_hidden, 1)

4. Step 4: b2 — bias for output layer\
b2 shape = (1, 1)\
self.b2 = np.zeros((1, 1))


np.zeros( shape )

And shape must be:

(128,) → 1D array

(3, 4) → 2D array

(2, 3, 4) → 3D array

In [36]:
import numpy as np

class TwoLayerNN:
  def __init__(self, n_inputs, n_hidden, learning_rate = 0.01, n_iters = 1000, seed =42):
    """
      n_inputs:  number of features (input dimension)
      n_hidden:  number of neurons in the hidden layer
      learning_rate: step size for gradient descent
      n_iters: number of training iterations
    """
    self.n_inputs = n_inputs
    self.n_hidden = n_hidden
    self.learning_rate = learning_rate
    self.n_iters = n_iters

    np.random.seed(seed)
    self.initialaize_params()
    self.loss_history = []

  #a helper method that initializes the weights and biases.
  def initialaize_params(self):
    """
      Initialize weights and biases:
      - W1: (n_inputs, n_hidden)
      - b1: (1, n_hidden)
      - W2: (n_hidden, 1)
      - b2: (1, 1)
      initialze weights with random value (to break symmetry) and bias with 0
      biases only shift the activation function
    """
    self.W1 = 0.01 * np.random.randn(self.n_inputs, self.n_hidden)
    self.b1 = np.zeros ((1, self.n_hidden))

    self.W2 = 0.01 * np.random.randn(self.n_hidden, 1)
    self.b2 = np.zeros((1,1)) # we create a tuple with ((,))

  # Activation functions:
  def relu(self, z):
    return np.maximum(0,z)

  def relu_derivative(self, z):
    # derivative of ReLU: 1 where z>0 else 0
    return (z>0).astype(float)

  def sigmoid(self, z):
    return 1/(1 + np.exp(-z))

  # forward pass:

  def forward_pass(self, X):
    """
        X: (n_samples, n_inputs)
        Returns:
          A2: output probabilities (n_samples, 1)
          cache: all intermediate values needed for backprop
    """
    # layer 1: ReLU
    Z1 = np.dot(X, self.W1) + self.b1
    A1 = self.relu(Z1)

    #layer 2: sigmoid
    Z2 = np.dot(A1, self.W2)+ self.b2
    A2 = self.sigmoid(Z2)

    cache = {
        "X": X,
        "Z1": Z1, "A1": A1,
        "Z2": Z2, "A2": A2
    }
    return A2, cache


  # loss function: (binary cross-entropy)
  """
  y_true: (n-samples, )
  y_pred: (n_samples, 1)
  """

  def compute_loss(self, y_true, y_pred):
    n_samples = y_true.shape[0] # number of training examples
    # here we don't have n_samples = X.shape because X can be sliced or reduced
    # avoid log(0)
    eps = 1e-15
    y_pred_clipped = np.clip(y_pred, eps, 1- eps)

    y_true = y_true.reshape(-1,1)
    """
    this is because:
    y_true.shape = (m,)
    y_pred.shape = (m, 1)

    """
    loss = -(1/n_samples) * np.sum(
        y_true * np.log(y_pred_clipped)+ (1 - y_true) * np.log(1- y_pred_clipped))

    return loss


  # backward pass (backpropagation)
  def backward(self, cache, y_true):
    """
      cache: dict from _forward
      y_true: (n_samples,)
      Computes gradients for W1, b1, W2, b2
      """
    m = y_true.shape[0] # m = n_samples
    y_true = y_true.reshape(-1,1)

    X = cache["X"]            # (m, n_inputs)
    Z1, A1 = cache["Z1"], cache["A1"]  # (m, n_hidden)
    Z2, A2 = cache["Z2"], cache["A2"]  # (m, 1)

    # Gradient of loss wrt Z2 (output pre-activation)
    # dL/dA2 = -(y/Â - (1-y)/(1-Â)), and for sigmoid + BCE this simplifies:
    dZ2 = (A2 - y_true)  # (m,1)

    # Gradients for W2 and b2
    dW2 = (1.0 / m) * np.dot(A1.T, dZ2)    # (n_hidden,1)
    db2 = (1.0 / m) * np.sum(dZ2, axis=0, keepdims=True)  # (1,1)

    # Backprop into hidden layer
    dA1 = np.dot(dZ2, self.W2.T)           # (m, n_hidden)
    dZ1 = dA1 * self.relu_derivative(Z1)  # (m, n_hidden)

    dW1 = (1.0 / m) * np.dot(X.T, dZ1)     # (n_inputs, n_hidden)
    db1 = (1.0 / m) * np.sum(dZ1, axis=0, keepdims=True) # (1, n_hidden)

    grads = {"dW1": dW1, "db1": db1,
              "dW2": dW2, "db2": db2}
    return grads

  # ---- Parameter update ----
  def update_params(self, grads):
    self.W1 -= self.learning_rate * grads["dW1"]
    self.b1 -= self.learning_rate * grads["db1"]
    self.W2 -= self.learning_rate * grads["dW2"]
    self.b2 -= self.learning_rate * grads["db2"]

  # ---- Training loop ----
  def fit(self, X, y, verbose=True):
    X = np.array(X, dtype=float)
    y = np.array(y, dtype=float)

    for i in range(self.n_iters):
        # 1. Forward
        y_pred, cache = self.forward_pass(X)

        # 2. Loss
        loss = self.compute_loss(y, y_pred)
        self.loss_history.append(loss)

        # 3. Backward
        grads = self.backward(cache, y)

        # 4. Update
        self.update_params(grads)

        # we either print at 100X X={1,2,3} or the last iteration
        if verbose and (i % 100 == 0 or i == self.n_iters - 1):
            print(f"Iteration {i:4d}  Loss: {loss:.4f}")

  # ---- Prediction helpers ----
  def predict_proba(self, X):
    X = np.array(X, dtype=float)
    y_pred, _ = self.forward_pass(X)
    return y_pred  # probabilities for class 1

  def predict(self, X, threshold=0.5):
    proba = self.predict_proba(X)
    return (proba >= threshold).astype(int).ravel()




np.clip(array, min_value, max_value)

x = np.array([0.0, 0.5, 1.0, 1.2, -0.1])\
np.clip(x, 0.01, 0.99)


In [37]:
import numpy as np

# Toy binary classification data (2D inputs)
np.random.seed(0)
n_samples = 200

# Class 0: centered at (0,0)
# // devides two numbers but removes decimal
X0 = np.random.randn(n_samples//2, 2) + np.array([-1, -1])
y0 = np.zeros(n_samples//2)

# Class 1: centered at (2,2)
X1 = np.random.randn(n_samples//2, 2) + np.array([2, 2])
y1 = np.ones(n_samples//2)

X = np.vstack([X0, X1])
y = np.concatenate([y0, y1])

# Shuffle
perm = np.random.permutation(n_samples)
X, y = X[perm], y[perm]

# Create and train the model
model = TwoLayerNN(n_inputs=2, n_hidden=8, learning_rate=0.1, n_iters=1000)
model.fit(X, y, verbose=True)

# Predictions
y_pred = model.predict(X)
accuracy = np.mean(y_pred == y)
print("Training accuracy:", accuracy)


Iteration    0  Loss: 0.6932
Iteration  100  Loss: 0.2623
Iteration  200  Loss: 0.0902
Iteration  300  Loss: 0.0676
Iteration  400  Loss: 0.0567
Iteration  500  Loss: 0.0526
Iteration  600  Loss: 0.0504
Iteration  700  Loss: 0.0490
Iteration  800  Loss: 0.0481
Iteration  900  Loss: 0.0475
Iteration  999  Loss: 0.0470
Training accuracy: 0.975
